<a href="https://colab.research.google.com/github/SaiGaneshChimmiri/Masai_Capstone_Project/blob/main/Module_1_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Required Libraries

In [16]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd
import sqlite3

#Web Scraping

In [17]:
def scrape_books(min_books=60, categories=3):
    url = "https://books.toscrape.com/"
    base_url = "https://books.toscrape.com/index.html"

    response = requests.get(base_url)
    soup = BeautifulSoup(response.text, "html.parser")

    category_tags = soup.select(".side_categories ul li ul li a")

    all_books = []
    categories_processed = 0

    for cat_tag in category_tags:
        if (
            categories_processed >= categories
            and len(all_books) >= min_books
        ):
            break

        category_name = cat_tag.text.strip()
        category_rel_url = cat_tag["href"]
        category_url = url + category_rel_url
        categories_processed += 1
        current_page_url = category_url
        while current_page_url:
            cat_response = requests.get(current_page_url)
            cat_response.raise_for_status()
            cat_soup = BeautifulSoup(cat_response.text, "html.parser")
            book_articles = cat_soup.find_all("article", class_="product_pod")

            for article in book_articles:
                title = article.h3.a["title"]
                price = article.find("p", class_="price_color").text.strip()

                rating_class = article.find("p", class_="star-rating")["class"]
                star_rating = [c for c in rating_class if c != "star-rating"][0]

                availability = (
                    article.find("p", class_="instock availability")
                    .text.strip()
                )

                all_books.append(
                    {
                        "title": title,
                        "price": price,
                        "star_rating": star_rating,
                        "availability": availability,
                        "category": category_name,
                    }
                )

            # Check for pagination next button
            next_button = cat_soup.select_one("li.next a")
            if next_button:
                next_href = next_button["href"]
                # Construct page URL relative to current category folder
                parent_dir = current_page_url.rsplit("/", 1)[0]
                current_page_url = parent_dir + "/" + next_href
            else:
                current_page_url = None

    return all_books


if __name__ == "__main__":
    books_data = scrape_books(min_books=60, categories=3)

    print(f"Successfully scraped {len(books_data)} books!\n")
    print("Sample Output (First 3 entries):\n")
    for book in books_data[:3]:
        print(book)

Successfully scraped 69 books!

Sample Output (First 3 entries):

{'title': "It's Only the Himalayas", 'price': 'Â£45.17', 'star_rating': 'Two', 'availability': 'In stock', 'category': 'Travel'}
{'title': 'Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', 'price': 'Â£49.43', 'star_rating': 'Four', 'availability': 'In stock', 'category': 'Travel'}
{'title': 'See America: A Celebration of Our National Parks & Treasured Sites', 'price': 'Â£48.87', 'star_rating': 'Three', 'availability': 'In stock', 'category': 'Travel'}


#Clean

In [18]:
import numpy as np
import pandas as pd


def clean_scraped_data(raw_books):
    df = pd.DataFrame(raw_books)

    # 1. Strip currency symbol (£) and convert price to float (price_gbp)
    def parse_price(val):
        try:
            val_clean = str(val).replace("£", "").replace("Â", "").strip()
            return float(val_clean)
        except (ValueError, TypeError):
            return np.nan

    df["price_gbp"] = df["price"].apply(parse_price)
    if df["price_gbp"].isnull().any():
        median_price = df["price_gbp"].median()
        df["price_gbp"].fillna(median_price, inplace=True)

    rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    df["rating"] = df["star_rating"].map(rating_map)
    if df["rating"].isnull().any():
        median_rating = int(df["rating"].median())
        df["rating"].fillna(median_rating, inplace=True)
    df["rating"] = df["rating"].astype(int)

    def parse_stock(val):
        if pd.isna(val):
            return False
        return "in stock" in str(val).lower()
    df["in_stock"] = df["availability"].apply(parse_stock)
    df = df.drop(columns=["price", "star_rating", "availability"])
    return df


if __name__ == "__main__":
    sample_raw_data = [
        {
            "title": "A Light in the Attic",
            "price": "£51.77",
            "star_rating": "Three",
            "availability": "In stock",
            "category": "Poetry",
        },
        {
            "title": "Tipping the Velvet",
            "price": "£53.74",
            "star_rating": "One",
            "availability": "In stock",
            "category": "Historical Fiction",
        },
        {
            "title": "Soumission",
            "price": "£50.10",
            "star_rating": "Four",
            "availability": "Out of stock",
            "category": "Fiction",
        },
    ]

    cleaned_df = clean_scraped_data(sample_raw_data)
    print(cleaned_df.to_string(index=False))

               title           category  price_gbp  rating  in_stock
A Light in the Attic             Poetry      51.77       3      True
  Tipping the Velvet Historical Fiction      53.74       1      True
          Soumission            Fiction      50.10       4     False


#Convert

In [19]:
def convert_gbp_to_inr(df: pd.DataFrame, rate: float = 105.50) -> pd.DataFrame:
    # conversion_rate (1 GBP = 105.50 INR)
    df["price_inr"] = (df["price_gbp"] * rate).round(2)
    return df


if __name__ == "__main__":
    sample_cleaned_data = pd.DataFrame(
        [
            {
                "title": "A Light in the Attic",
                "category": "Poetry",
                "price_gbp": 51.77,
                "rating": 3,
                "in_stock": True,
            },
            {
                "title": "Tipping the Velvet",
                "category": "Historical Fiction",
                "price_gbp": 53.74,
                "rating": 1,
                "in_stock": True,
            },
            {
                "title": "Soumission",
                "category": "Fiction",
                "price_gbp": 50.10,
                "rating": 4,
                "in_stock": False,
            },
        ]
    )


    final_df = convert_gbp_to_inr(sample_cleaned_data, rate=105.50)

    print(final_df[["title", "price_gbp", "price_inr"]])

                  title  price_gbp  price_inr
0  A Light in the Attic      51.77    5461.74
1    Tipping the Velvet      53.74    5669.57
2            Soumission      50.10    5285.55


#Store

In [23]:
def populate_query_and_verify(
    cleaned_df: pd.DataFrame, db_name="books_database.db"
):
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    cursor.execute("PRAGMA foreign_keys = ON;")

    unique_categories = cleaned_df["category"].unique()
    for cat in unique_categories:
        cursor.execute(
            "INSERT OR IGNORE INTO categories (category_name) VALUES (?)",
            (cat,),
        )
    conn.commit()

    cat_map_df = pd.read_sql_query(
        "SELECT category_id, category_name FROM categories", conn
    )
    cat_to_id = dict(
        zip(cat_map_df["category_name"], cat_map_df["category_id"])
    )

    books_data = cleaned_df.copy()
    books_data["category_id"] = books_data["category"].map(cat_to_id)
    books_data["in_stock"] = books_data["in_stock"].astype(int)

    books_to_insert = books_data[
        [
            "title",
            "price_gbp",
            "price_inr",
            "rating",
            "in_stock",
            "category_id",
        ]
    ].values.tolist()

    cursor.executemany(
        """
        INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """,
        books_to_insert,
    )
    conn.commit()

    print("Successfully populated normalized SQLite database!\n")


    q1 = "SELECT title, price_gbp, rating FROM books WHERE in_stock = 1 ORDER BY price_gbp DESC LIMIT 5;"
    q2 = "SELECT DISTINCT rating FROM books ORDER BY rating ASC;"
    q3 = "SELECT title, price_inr, rating FROM books WHERE price_inr BETWEEN 2000 AND 5000 ORDER BY price_inr ASC LIMIT 5;"
    q4 = "SELECT title, price_gbp, rating FROM books WHERE rating IN (4, 5) AND in_stock = 1 LIMIT 5;"
    q5 = """
        SELECT b.title, c.category_name, b.price_gbp, b.price_inr, b.rating
        FROM books b
        JOIN categories c ON b.category_id = c.category_id
        WHERE b.rating >= 4
        ORDER BY b.price_gbp DESC
        LIMIT 10;
    """

    queries = [
        ("Query 1 (SELECT/WHERE/ORDER BY/LIMIT)", q1),
        ("Query 2 (DISTINCT)", q2),
        ("Query 3 (BETWEEN)", q3),
        ("Query 4 (IN)", q4),
        ("Query 5 (JOIN)", q5),
    ]

    for title, q_str in queries:
        print(f"=== {title} ===")
        res_df = pd.read_sql_query(q_str, conn)
        print(res_df.to_string(index=False))
        print("-" * 60 + "\n")



    sql_join_result = pd.read_sql_query(q5, conn)

    db_books = pd.read_sql_query("SELECT * FROM books", conn)
    db_categories = pd.read_sql_query("SELECT * FROM categories", conn)

    pandas_merged = (
        pd.merge(
            db_books,
            db_categories,
            on="category_id",
            how="inner",
        )
        .query("rating >= 4")
        .sort_values(by="price_gbp", ascending=False)
        .head(10)[["title", "category_name", "price_gbp", "price_inr", "rating"]]
        .reset_index(drop=True)
    )

    print("SQL Join Result shape:", sql_join_result.shape)
    print("Pandas Merge Result shape:", pandas_merged.shape)
    print(
        "Are both results identical?:",
        sql_join_result.equals(pandas_merged),
    )

    conn.close()
